# Week 3 Study Guide -- Layer by Layer

This guide is a **gentle tour** of the building blocks you will see in Sessions 5 and 6. It is the first time you are meeting Neural Networks, so we go slowly:

1. **Layer reference card** -- what each layer is and what it does.
2. **Build a Dense Neural Network**, one layer at a time, with comments on every line.
3. **Build a Convolutional Neural Network**, the same way.
4. **Meet the Vision Transformer** -- a different kind of network that does **not** use convolution.
5. **Patterns for each TODO** -- short, copy-friendly recipes you can reuse.

There are **no helper functions** in this notebook. Every model is written out from top to bottom, so you can read it like a recipe.

In [ ]:
# Imports + reproducibility -- run this first
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.optimizers import Adam
from scipy.signal import convolve2d

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)

In [ ]:
# Load a TINY CIFAR-10 subset so every demo is instant
(X_full, y_full), _ = cifar10.load_data()

X_demo = X_full[:200].astype('float32') / 255.0   # 200 images, scaled to 0-1
y_demo = y_full[:200].flatten()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'X_demo shape: {X_demo.shape}')
print(f'y_demo shape: {y_demo.shape}')

## Layer Reference Card

Every line you add to a Keras model is `layers.Something(...)`. But there are **two very different kinds** of "layers":

- **Real (learning) layers** -- have weights, do the actual learning. `Dense`, `Conv2D`.
- **Plumbing pieces** -- shape changes or regularisation. **No weights, no learning.** `Input`, `Flatten`, `MaxPooling2D`, `AveragePooling2D`, `Dropout`.

When people count "the layers of the network" they usually mean the **learning** ones.

| Piece | Has weights? | What it does | What you usually set |
|-------|:---:|--------------|----------------------|
| `Input(shape=...)`        | NO  | Tells Keras the shape of **one** input. Always first. Not really a layer -- think of it as a label on the front door. | `shape=(32, 32, 3)` for a colour image |
| `Flatten()`               | NO  | Stretches a 2D or 3D array into one long 1D row. Pure shape change. | nothing |
| **`Dense(units, ...)`**       | **YES** | A **fully connected** learning layer. Every input is connected to every neuron. | `units` (how many neurons), `activation` |
| **`Conv2D(filters, ...)`**    | **YES** | A learning layer that slides small filters over an image to find local patterns (edges, textures, shapes). | `filters`, `kernel_size`, `activation='relu'`, `padding='same'` |
| `MaxPooling2D(pool_size)` | NO  | Shrinks an image by keeping the **largest** value in each small block. | `pool_size=2` (halves the image) |
| `AveragePooling2D(pool_size)` | NO | Same idea but takes the **average** instead. | `pool_size=2` |
| `Dropout(rate)`           | NO  | During training only, randomly switches off `rate` fraction of inputs. Fights overfitting. | `rate=0.5` is common |

**Activations in plain English:**
- `relu`  -- `max(0, x)`. Throws away negatives, keeps positives.
- `sigmoid` -- squashes any number to between 0 and 1. The classic before ReLU.
- `softmax` -- converts the final N numbers into probabilities that add up to 1. Use it on the output layer of a classifier.

## Part 1 -- Build a Dense Neural Network, layer by layer

A Dense Neural Network treats every pixel as an independent number. This network has **only two learning layers** (the two `Dense` lines below). The `Input` and `Flatten` lines are plumbing -- they describe and reshape the input but do no learning.

The recipe:

1. Start with a `Sequential` container.
2. Describe the input + reshape it (plumbing).
3. **Add the learning layers**, one at a time.
4. Compile so Keras knows how to train it.

In [ ]:
# Create the empty container
model_dense = models.Sequential()

# === Input setup (no learning happens here -- these just describe and reshape the data) ===
# describe one image: 32 wide, 32 tall, 3 colour channels
model_dense.add(layers.Input(shape=(32, 32, 3)))
# stretch the 32x32x3 cube into a single 3072-long row, because Dense layers want 1D input
model_dense.add(layers.Flatten())

# === Learning Layer 1 -- hidden Dense ===
# 128 neurons, each connected to all 3072 inputs. ReLU activation = max(0, x).
model_dense.add(layers.Dense(128, activation='relu'))

# === Learning Layer 2 -- output Dense ===
# 10 neurons, one per class. Softmax turns the 10 numbers into probabilities that sum to 1.
model_dense.add(layers.Dense(10, activation='softmax'))

# Compile -- tell Keras how to train this:
#   * how to update weights      -> optimizer = 'adam'
#   * how to score wrongness     -> loss = 'sparse_categorical_crossentropy'
#   * what to display each epoch -> metrics = ['accuracy']
model_dense.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

model_dense.summary()
# Notice: only the two Dense rows have non-zero "Param #" -- those are the learning layers.

## Part 2 -- Build a CNN, layer by layer

A Convolutional Neural Network respects the **2D shape** of an image. This network has **four learning layers** -- two `Conv2D` and two `Dense`. The `Input`, `MaxPooling2D` and `Flatten` lines are plumbing pieces (no weights, no learning).

The standard CNN recipe is **(Conv -> Pool) -> (Conv -> Pool) -> Flatten -> small Dense head**.

In [ ]:
# Create the empty container
model_cnn = models.Sequential()

# === Input setup (no learning) ===
model_cnn.add(layers.Input(shape=(32, 32, 3)))

# === Learning Layer 1 -- Conv2D + downsampler ===
# Conv2D(32, 3) means: learn 32 small filters, each 3x3 pixels.
# padding='same' keeps the output the same width/height as the input.
model_cnn.add(layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'))
# MaxPooling halves the image (no weights, no learning -- just shrinks the data)
model_cnn.add(layers.MaxPooling2D(pool_size=2))

# === Learning Layer 2 -- deeper Conv2D ===
# Deeper layers usually have more filters. Each filter combines patterns the earlier filters found.
model_cnn.add(layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'))
model_cnn.add(layers.MaxPooling2D(pool_size=2))   # shrink again (no learning)

# === Switch from 2D to 1D (no learning) ===
# After 2 pooling steps the image is 8x8x64. Flatten it for the dense head.
model_cnn.add(layers.Flatten())

# === Learning Layer 3 -- Dense head ===
# Combines the conv features into something the output layer can use.
model_cnn.add(layers.Dense(64, activation='relu'))

# === Learning Layer 4 -- output Dense ===
# 10 classes, softmax for probabilities.
model_cnn.add(layers.Dense(10, activation='softmax'))

# Compile (same recipe as the dense model).
model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

model_cnn.summary()
# Notice: only the four "learning" rows (2x Conv2D, 2x Dense) have non-zero "Param #".

## Part 3 -- Meet the Vision Transformer (ViT)

A **Vision Transformer** is a different family of network. It does **not** use `Conv2D` at all. We are not going to build one from scratch in this course -- transformers usually need millions of images to train well. Instead, we use a **pre-trained** ViT from Hugging Face (it already learned on ImageNet) and feed our images into it.

### How does ViT "see" an image?

A CNN slides small filters across the image looking for local patterns. A transformer is different:

1. **Cut the image into patches.** A 224x224 image is sliced into a grid of 16x16 patches -- so 14x14 = 196 patches in total.
2. **Turn each patch into a vector** of numbers (an "embedding"). Each patch becomes one row.
3. **Self-attention.** The model asks, for every patch, *"how much should I pay attention to every other patch?"* This is the core idea -- patches can compare themselves to all the others, no matter how far apart they are. There is no sliding window.
4. **Classify.** A final dense layer turns the result into class probabilities.

### CNN vs ViT -- side by side

| Question | CNN | Vision Transformer |
|----------|-----|---------------------|
| How does it look at the image?     | Slides small filters across the pixels | Splits into patches and attends globally |
| Maths inside                       | Convolution + pooling                  | Self-attention (matrix multiplications)  |
| Train from scratch on small data?  | Yes -- works fine                      | No -- usually needs millions of images   |
| Best when...                       | Small dataset, fast training           | A good pretrained model is available     |

You will not write a transformer in this course. **Pattern I** at the end of this guide shows you how to *use* one in two lines of code via Hugging Face.

## Part 4 -- Patterns for the TODOs

Each pattern below is a small recipe you can copy into the session notebook. Each one tells you exactly which TODO it matches.

### Pattern A -- Filter images by class and plot a grid

When you need to show several images of one specific class:

1. Use `np.where(y_demo == chosen_class)` to find the indices of that class.
2. Take the first few with `[:8]`.
3. Plot them with `plt.subplots(2, 4)` and an `axes.flat` loop.

**Use this in:** _Session 5 -- TODO 1.1_

In [ ]:
# Filter for class 5 ('dog') and plot 8 examples
chosen_class = 5
idx = np.where(y_demo == chosen_class)[0][:8]    # first 8 matching indices

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, idx):
    ax.imshow(X_demo[i])
    ax.set_title(class_names[y_demo[i]], fontsize=9)
    ax.axis('off')
plt.suptitle(f"8 images of class '{class_names[chosen_class]}'")
plt.tight_layout()
plt.show()

### Pattern B -- Swap one argument (e.g. activation)

The whole model only differs by **one word**. Below we build the same dense network but with `sigmoid` instead of `relu`. To try any other change, copy this cell and edit the line you care about.

**Use this in:** _Session 5 -- TODO 1.2_

In [ ]:
# Same model as model_dense, but with sigmoid in the hidden Dense layer
model_sigmoid = models.Sequential()

# Input setup (no learning)
model_sigmoid.add(layers.Input(shape=(32, 32, 3)))
model_sigmoid.add(layers.Flatten())

# Learning Layer 1 -- hidden Dense with SIGMOID this time
model_sigmoid.add(layers.Dense(128, activation='sigmoid'))    # <-- this is the only change

# Learning Layer 2 -- output Dense (unchanged)
model_sigmoid.add(layers.Dense(10,  activation='softmax'))

model_sigmoid.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

print('Same architecture, different activation.')
print('relu params:   ', model_dense.count_params())
print('sigmoid params:', model_sigmoid.count_params())

### Pattern C -- Forward pass through a Dense layer by hand

A Dense layer does three things:

1. Multiply input by weights:  `z = x @ W + b`
2. Apply the activation:        `a = max(0, z)`  for ReLU

To do it yourself: grab the layer's weights with `.get_weights()`, flatten one image, and run the maths.

**Use this in:** _Session 5 -- TODO 2.1_

In [ ]:
# Step 1 -- get the weights of the FIRST dense layer of model_dense
#   layer 0 = Flatten, layer 1 = Dense(128). get_weights() returns [W, b].
W1, b1 = model_dense.layers[1].get_weights()
print(f'W1 shape: {W1.shape}    (3072 inputs -> 128 neurons)')
print(f'b1 shape: {b1.shape}')

# Step 2 -- pick an image and flatten it to a 1D vector of 3072 numbers
img_idx = 7
x = X_demo[img_idx].reshape(-1)
print(f'\ninput shape: {x.shape}')

# Step 3 -- the maths: weighted sum + bias, then ReLU
z1 = x @ W1 + b1
a1 = np.maximum(0, z1)

print(f'output shape: {a1.shape}')
print(f'first 5 activations: {a1[:5].round(3)}')

### Pattern D -- Try a learning rate

To try different learning rates, **copy this cell** and change the number on the highlighted line. The TODO asks for three values (`0.001`, `0.1`, `10.0`) -- just run the cell three times, saving the loss curve into a different variable each time.

**Use this in:** _Session 5 -- TODO 2.2_

In [ ]:
# Reset seeds so different runs are comparable
np.random.seed(42); tf.random.set_seed(42)

# Build a small dense model from scratch
m = models.Sequential()

# Input setup (no learning)
m.add(layers.Input(shape=(32, 32, 3)))
m.add(layers.Flatten())

# Two learning layers
m.add(layers.Dense(64, activation='relu'))
m.add(layers.Dense(10, activation='softmax'))

# Compile -- this is where the learning rate lives
m.compile(
    optimizer=Adam(learning_rate=0.001),    # <-- copy this cell and change this number
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Train for 1 epoch on the demo data (use 3 epochs in the real TODO)
history = m.fit(X_demo, y_demo, epochs=1, batch_size=32, verbose=0)

# Plot the loss curve from this run
plt.plot(history.history['loss'], marker='o')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('learning rate = 0.001')
plt.show()

# To overlay 3 runs in the real TODO:
#   loss_001 = m.fit(... lr=0.001 ...).history['loss']
#   loss_01  = m.fit(... lr=0.1   ...).history['loss']
#   loss_10  = m.fit(... lr=10.0  ...).history['loss']
# then plt.plot(loss_001); plt.plot(loss_01); plt.plot(loss_10)

### Pattern E -- Apply a 2D kernel to an image

A kernel is just a small matrix of numbers. `convolve2d` slides it across the image and computes a weighted sum at every position. Use `mode='same'` so the output is the same size as the input.

The image must be **2D** (greyscale), so we average the RGB channels first.

**Use this in:** _Session 5 -- TODO 3.1_

In [ ]:
# Make a greyscale version of one image (average the 3 colour channels)
gray = X_demo[0].mean(axis=2)

# Define three classic kernels
edge_kernel = np.array([[-1, -1, -1],
                        [-1,  8, -1],
                        [-1, -1, -1]])

blur_kernel = np.ones((3, 3)) / 9.0     # average of the 9 neighbours

sharpen_kernel = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])

# Apply each kernel
edges    = convolve2d(gray, edge_kernel,    mode='same', boundary='symm')
blurred  = convolve2d(gray, blur_kernel,    mode='same', boundary='symm')
sharpened = convolve2d(gray, sharpen_kernel, mode='same', boundary='symm')

# Show all four side by side
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(gray,      cmap='gray'); axes[0].set_title('original');  axes[0].axis('off')
axes[1].imshow(edges,     cmap='gray'); axes[1].set_title('edges');     axes[1].axis('off')
axes[2].imshow(blurred,   cmap='gray'); axes[2].set_title('blurred');   axes[2].axis('off')
axes[3].imshow(sharpened, cmap='gray'); axes[3].set_title('sharpened'); axes[3].axis('off')
plt.tight_layout()
plt.show()

### Pattern F -- Predict and read the top-K classes

`model.predict` needs a **batch**, so use `X[i:i+1]` (with the colon!) instead of `X[i]`. The output has 10 numbers (one probability per class). To get the top 3, sort the indices and take the last three reversed.

**Use this in:** _Session 5 -- TODO 3.2_

In [ ]:
# Pick any image from the demo set
img_idx = 7

# Step 1 -- predict (note the i:i+1 to keep the batch dimension)
probs = model_dense.predict(X_demo[img_idx:img_idx+1], verbose=0)[0]   # shape (10,)

# Step 2 -- find the top 3 indices, highest first
top3 = probs.argsort()[-3:][::-1]

# Step 3 -- print a nice ranking
print(f"actual: {class_names[y_demo[img_idx]]}\n")
print('top-3 predictions:')
for rank, c in enumerate(top3, 1):
    print(f'  {rank}. {class_names[c]:12s}  p = {probs[c]:.3f}')

### Pattern G -- Visualise feature maps

A **feature map** is the output of one conv layer for one image. To see them, build a tiny sub-model whose output is the conv layer you want, then `predict` on a single image. The result has shape `(1, height, width, num_filters)` -- one feature map per filter.

**Use this in:** _Session 6 -- TODO 1.1_

In [ ]:
# Step 1 -- build a sub-model whose OUTPUT is the first conv layer of model_cnn
#   model_cnn.layers[0] is the first Conv2D (Input is hidden inside Sequential).
feature_extractor = tf.keras.Model(inputs=model_cnn.inputs,
                                   outputs=model_cnn.layers[0].output)

# Step 2 -- run it on ONE image (remember the i:i+1 trick to keep the batch dim)
fmaps = feature_extractor.predict(X_demo[0:1], verbose=0)
print(f'feature maps shape: {fmaps.shape}   (1 image, 32x32, 32 filters)')

# Step 3 -- plot the first 8 feature maps
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmaps[0, :, :, i], cmap='viridis')
    ax.set_title(f'filter {i}', fontsize=9)
    ax.axis('off')
plt.suptitle('First conv layer -- feature maps for one image')
plt.tight_layout()
plt.show()

### Pattern H -- Change ONE thing in the CNN

For Session 6 Activity 2 you will rebuild the CNN with **one thing different each time** (different filter count, different kernel size, with dropout, deeper, ...). The cell below is a copy-friendly template -- the comments mark every line you might want to change. **Just copy this cell and edit one or two lines.**

**Use this in:** _Session 6 -- TODO 2.1, 2.2, 2.3_

In [ ]:
# Reset seeds so two runs are fair to compare
np.random.seed(42); tf.random.set_seed(42)

m_variant = models.Sequential()

# === Input setup (no learning) ===
m_variant.add(layers.Input(shape=(32, 32, 3)))

# === Learning Layer 1 -- Conv2D ===
#   - first number is the FILTER COUNT (try 8, 16, 32, 64)
#   - kernel_size is the KERNEL SIZE  (try 3, 5, 7)
m_variant.add(layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'))
#   - swap MaxPooling2D for AveragePooling2D to test pooling type (still no learning)
m_variant.add(layers.MaxPooling2D(pool_size=2))

# === Learning Layer 2 -- Conv2D ===
m_variant.add(layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'))
m_variant.add(layers.MaxPooling2D(pool_size=2))

# === Optional Learning Layer 3 -- DEEPER ===
# Uncomment these two lines to add a third conv block.
# m_variant.add(layers.Conv2D(128, kernel_size=3, activation='relu', padding='same'))
# m_variant.add(layers.MaxPooling2D(pool_size=2))

# === Switch from 2D to 1D (no learning) ===
m_variant.add(layers.Flatten())

# === Learning Layer 3 (or 4 if you uncommented above) -- Dense head ===
m_variant.add(layers.Dense(64, activation='relu'))

# === Optional regularisation (no learning, but kicks in during training only) ===
# Uncomment to add dropout (try 0.3 or 0.5).
# m_variant.add(layers.Dropout(0.5))

# === Output Dense ===
m_variant.add(layers.Dense(10, activation='softmax'))

m_variant.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print('Total parameters:', m_variant.count_params())
m_variant.summary()

### Pattern I -- Run a Hugging Face image-classification pipeline

Vision Transformer (ViT) was trained on 224x224 images, so we have to resize first. The recipe:

1. NumPy 32x32 image (0-1 floats) -> `(image * 255).astype('uint8')`
2. `Image.fromarray(...)` to make a PIL image
3. `.resize((224, 224))` to match ViT's input size
4. Pass the PIL image into the `pipeline(...)`. It returns a list of `{'label': ..., 'score': ...}` dicts.

> The first call downloads ~350 MB. The cell below shows the full recipe with the model load **commented out** so this guide stays fast. Uncomment when you actually want to predict.

**Use this in:** _Session 6 -- TODO 3.1_

In [ ]:
from PIL import Image
from transformers import pipeline

# Step 1 -- pick an image from the demo set
img_idx = 7

# Step 2 -- numpy (0-1 floats) -> numpy (0-255 uint8)
np_img = (X_demo[img_idx] * 255).astype('uint8')

# Step 3 -- numpy -> PIL, then resize to 224x224
pil_img = Image.fromarray(np_img).resize((224, 224))

# Sanity check -- show the image we are about to feed to ViT
plt.imshow(pil_img); plt.axis('off')
plt.title(f'CIFAR image {img_idx} resized to 224x224')
plt.show()

# Step 4 -- run ViT.
#           UNCOMMENT the two lines below the FIRST time you run this --
#           the model download is ~350 MB and takes ~30 seconds.
#
# vit  = pipeline('image-classification', model='google/vit-base-patch16-224', top_k=3)
# preds = vit(pil_img)
# for p in preds:
#     print(f"  {p['label']:30s}  score={p['score']:.3f}")
print('(uncomment the vit lines above to run the actual prediction)')

## Quick Reference -- find your pattern fast

| Session | TODO | Pattern in this guide |
|---------|------|------------------------|
| Session 5 | TODO 1.1 -- 8 images of one class       | Pattern A |
| Session 5 | TODO 1.2 -- swap activation             | Pattern B |
| Session 5 | TODO 2.1 -- forward pass on new image   | Pattern C |
| Session 5 | TODO 2.2 -- learning-rate sweep         | Pattern D |
| Session 5 | TODO 3.1 -- blur + sharpen kernels      | Pattern E |
| Session 5 | TODO 3.2 -- predict on your own image   | Pattern F |
| Session 6 | TODO 1.1 -- visualise feature maps      | Pattern G |
| Session 6 | TODO 2.1, 2.2, 2.3 -- hyperparameters   | Pattern H |
| Session 6 | TODO 3.1 -- run ViT on your image       | Pattern I |

**Stuck on a TODO?** Find the matching pattern, copy the cell into a scratch cell in your session notebook, get it running there first, then merge it into your TODO.